In [4]:
import pandas as pd
import random
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    SentenceTransformerModelCardData,
)
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers
from datasets import Dataset

train_df = pd.read_excel("train_set.xlsx")
val_df = pd.read_excel("val_set.xlsx")
test_df = pd.read_excel("test_set.xlsx")

# Keep only positives (Annotation == 2 in your case)
train_pos = train_df[train_df["Annotation"] == 2]
val_pos = val_df[val_df["Annotation"] == 2]
test_pos = test_df[test_df["Annotation"] == 2]

# Build dicts: {Question: [list of positive answers]}
train_q2a = train_pos.groupby("Question")["Content"].apply(list).to_dict()
val_q2a = val_pos.groupby("Question")["Content"].apply(list).to_dict()
test_q2a = test_pos.groupby("Question")["Content"].apply(list).to_dict()


# --------------------------
# Function to build HuggingFace Dataset with cycling
# --------------------------
def build_pairs(question2answers, num_epochs=3):
    """
    Build list of (q, a) pairs where each positive answer is seen
    once per cycle. If a question has multiple positives, they are all
    included across epochs.
    """
    pairs = []
    for _ in range(num_epochs):
        for q, answers in question2answers.items():
            for a in answers:
                pairs.append({"texts": [q, a]})
    return pairs


# --------------------------
# Create datasets
# --------------------------
train_pairs = build_pairs(train_q2a, num_epochs=3)  # cycle multiple times
val_pairs = build_pairs(val_q2a, num_epochs=1)  # usually 1 cycle for eval
test_pairs = build_pairs(test_q2a, num_epochs=1)

train_dataset = Dataset.from_list(train_pairs)
val_dataset = Dataset.from_list(val_pairs)
test_dataset = Dataset.from_list(test_pairs)

print(train_dataset)



Dataset({
    features: ['texts'],
    num_rows: 768
})


In [5]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print(
        "Memory total (MB):", torch.cuda.get_device_properties(0).total_memory / 1024**2
    )
    print("Memory allocated (MB):", torch.cuda.memory_allocated(0) / 1024**2)
    print("Memory reserved (MB):", torch.cuda.memory_reserved(0) / 1024**2)


CUDA available: True
Device count: 1
GPU name: NVIDIA RTX A6000
Memory total (MB): 48669.75
Memory allocated (MB): 0.0
Memory reserved (MB): 0.0
